In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only ".
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Imports

In [2]:
!pip install transformers

In [3]:
from transformers import AutoModelForAudioClassification,AutoFeatureExtractor
from torch.utils.data import Dataset,DataLoader
import os
import torch.nn as nn
import torch
from sklearn.model_selection import train_test_split
import librosa
import wandb
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import f1_score
import glob
import random

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("login")
os.environ["WANDB_API_KEY"]=wandb_api
wandb.login()
print("Wandb login successful")

wandb: Currently logged in as: 23f1000559 (23f1000559-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb login successful


## Model

In [5]:
model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"

In [6]:
model = AutoModelForAudioClassification.from_pretrained(
    model_name, num_labels = 10 , ignore_mismatched_sizes=True
                                                       )
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)



config.json: 0.00B [00:00, ?B/s]

2026-03-09 05:34:49.650002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773034489.884240      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773034489.949590      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773034490.505890      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773034490.505933      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773034490.505936      24 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [7]:
# freeze all the layers
for param in model.parameters():
    param.requires_grad=False
    
# unfreeze last 6 transformer blocks and classification layer
for layer in model.audio_spectrogram_transformer.encoder.layer[-6:]:
    for param in layer.parameters():
        param.requires_grad = True
        
for param in model.classifier.parameters():
    param.requires_grad=True
    
    

## Load Dataset

In [8]:
!ls /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00000

bass.wav  drums.wav  other.wav	vocals.wav


In [9]:
# collect song directories 
song_dirs=[]
labels = []
genres = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]
root_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
for genre_idx, genre in enumerate(genres):
    genre_path = os.path.join(root_dir, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        song_dirs.append(song_path)
        labels.append(genre_idx)

In [10]:
#read  noise files 
noise_files = glob.glob("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/*.wav")


In [11]:
class MyDataset(Dataset):
    def __init__(self,song_dirs,labels,noise_files,feature_extractor,sr=16000,duration=10,augment=True):
        self.song_dirs = song_dirs
        self.labels = labels
        self.noise_files = noise_files
        self.feature_extractor = feature_extractor
        self.sr = sr
        self.max_len = sr*duration
        self.augment = augment
        # group songs with their genres
        self.genre_dirs={}
        for d , l in zip(song_dirs,labels):
            self.genre_dirs.setdefault(l,[]).append(d)
        
    def pad_truncate(self,audio):
        if len(audio) > self.max_len :
            audio = audio[:self.max_len]
        else: 
            audio = np.pad(audio,(0,(self.max_len-len(audio))))
        return audio
    def load_stem(self,song_dir,stem_name):
        candidates = [stem_name, stem_name.replace("other.wav", "others.wav")]
        for name in candidates:
            path = os.path.join(song_dir, name)
            if os.path.exists(path):
                audio,_ = librosa.load(path,sr=self.sr)
                return self.pad_truncate(audio)
        return np.zeros(self.max_len)
    def add_noise(self,audio):
        noise,_ = librosa.load(random.choice(self.noise_files),sr=self.sr)
        noise = self.pad_truncate(noise)
        noise_level = np.random.uniform(0.05,0.3)
        return audio + noise*noise_level
        
    def __len__(self):
        return len(self.song_dirs)
    def __getitem__(self,idx):
        label = self.labels[idx]
        stems = ["bass.wav", "drums.wav", "vocals.wav", "other.wav"]
        if self.augment:
            # different songdir for every stem from same genre
            same_genre_dirs = self.genre_dirs[label]
            stem_audio=[]
            for stem in stems:
                ran_dir = random.choice(same_genre_dirs)
                stem_audio.append(self.load_stem(ran_dir,stem))
            # random stem weights
            weights = np.random.uniform(0.5,1.5,4)
            mix = sum(w*a for w,a in zip(weights,stem_audio))
            #tempo stretch 
            if np.random.rand()<0.5:
                rate = np.random.uniform(0.85,1.15)
                mix = librosa.effects.time_stretch(mix,rate=rate)
                mix = self.pad_truncate(mix)
            # add noise
            mix = self.add_noise(mix)
        else:
            #val no augmentation
            mix = sum(self.load_stem(self.song_dirs[idx],s) for s in stems)
        # normailise
        mix = mix / (np.max(np.abs(mix)) + 1e-6)
        inputs = self.feature_extractor(
            mix, sampling_rate=self.sr, return_tensors="pt"
        )
        return inputs["input_values"].squeeze(0), torch.tensor(label)


In [12]:
# train val split
train_files,val_files,train_labels,val_labels = train_test_split(song_dirs,labels,random_state=42,test_size=0.2,stratify=labels)

In [13]:
# dataset instance 
train_dataset = MyDataset(train_files,train_labels,noise_files,feature_extractor,augment=True)
val_dataset = MyDataset(val_files,val_labels,noise_files,feature_extractor,augment=False)

In [14]:
#dataloaders
train_loader = DataLoader(train_dataset,batch_size=4,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=4,shuffle=False)

## Training Loop

In [15]:
# loss and optimisers
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optim = torch.optim.AdamW([
    {"params": model.classifier.parameters(), "lr": 1e-3},
    {"params": model.audio_spectrogram_transformer.encoder.layer[-6:].parameters(), "lr": 1e-5},
])

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [17]:
# Start a new wandb run 
run = wandb.init(
    # Set the wandb project where this run will be logged.
    project="23f1000559-t12026",
    entity ="23f1000559-indian-institute-of-technology-madras",
    # Track hyperparameters and run metadata.
    config={
        "epochs": 12,
        "batch_size": 4,
        "learning_rate": [1e-3 ,1e-5],
        "model": "MIT/ast-finetuned-audioset-v6"
    },
)

wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260309_053508-jnsrl66f
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run worthy-field-13
wandb: ⭐️ View project at https://wandb.ai/23f1000559-indian-institute-of-technology-madras/23f1000559-t12026
wandb: 🚀 View run at https://wandb.ai/23f1000559-indian-institute-of-technology-madras/23f1000559-t12026/runs/jnsrl66f


In [18]:
model.to(device)
num_epochs = 12
for epoch in range(num_epochs):
    total_loss = 0
    model.train()
    for x,y in train_loader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        logits = output.logits
        loss = criterion(logits,y)
        optim.zero_grad()
        loss.backward()
        optim.step()
        total_loss +=loss.item()
    avg_loss = loss/len(train_loader)
    #Log loss to wandb
    wandb.log({"epoch": epoch+1, "train_loss": avg_loss})
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    

Epoch 1/12, Loss: 0.0034
Epoch 2/12, Loss: 0.0031
Epoch 3/12, Loss: 0.0096
Epoch 4/12, Loss: 0.0030
Epoch 5/12, Loss: 0.0039
Epoch 6/12, Loss: 0.0036
Epoch 7/12, Loss: 0.0028
Epoch 8/12, Loss: 0.0030
Epoch 9/12, Loss: 0.0034
Epoch 10/12, Loss: 0.0030
Epoch 11/12, Loss: 0.0028
Epoch 12/12, Loss: 0.0028


## Validation Loop

In [19]:
model.eval()
total_val_loss =0
all_preds=[]
all_labels = []
with torch.no_grad():
    for x_val,y_val in val_loader:
        x_val = x_val.to(device)
        y_val = y_val.to(device)
        output= model(x_val)
        logits = output.logits
        loss = criterion(logits, y_val)
        total_val_loss += loss.item()
        preds = logits.argmax(dim=1)
        # store for f1
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_val.cpu().numpy())
avg_val_loss = total_val_loss / len(val_loader)
val_macro_f1 = f1_score(all_labels,all_preds,average="macro")
print(f"Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
#log to wandb
wandb.log({"val_loss": avg_val_loss, "val_macro_f1": val_macro_f1})
        

Val Loss: 0.9746, Val Macro F1: 0.8177


## Submission

In [20]:
genres = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
idx_genre = {i: g for i , g in enumerate(genres)}

In [21]:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

In [22]:
max_len = 160000
def pad_truncate(audio, max_len=max_len):
    if len(audio) > max_len:
        audio = audio[:max_len]
    else:
        pad_len = max_len - len(audio)
        audio = np.pad(audio, (0, pad_len))
    return audio
def predict_genre(file_path, model, feature_extractor, device):
    audio, sr = librosa.load(file_path, sr=16000)
    audio = pad_truncate(audio)
    
    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
    input_values = inputs["input_values"].to(device)
    with torch.no_grad():
        output = model(input_values)
        logits = output.logits
        pred_idx = logits.argmax(dim=1).item()
    return idx_genre[pred_idx]

In [23]:
predictions = []
model.eval()
model.to(device)
for file_path in test_df["filename"]:
    root = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    final_path = os.path.join(root,file_path)
    genre_pred = predict_genre(final_path, model, feature_extractor, device)
    predictions.append(genre_pred)

In [24]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.head())

   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
